# Session 5 · The Same Pipeline in PySpark

**Goal:** rebuild sessions 2-4 in PySpark, prove it agrees with pandas to the cent, and save the result as a governed table.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

**pandas vs Spark in one sentence:** pandas works on one machine and runs each line immediately; Spark splits work across a cluster and only runs when you ask for a result (a count, a display, or a write). Same ideas, different engine.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 5

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
from pyspark.sql import functions as F
from lnl import check, cleaning, conversions, domains, pipeline
from lnl import spark as lspark
from lnl.check import ___

spark = lspark.get_spark()
raw = lspark.read_raw(spark)
print(f"{raw.count():,} rows")
raw.printSchema()

Spark's CSV reader keeps every column as text by default (so IDs arrive as 8-digit text, leading zeros intact) and reads truly empty fields as null. Values that are only spaces or tabs survive as text, same as in pandas.

## 1. Session 2 in Spark
First, a trap. Spark has a `trim()` function. Watch what it does:

In [ ]:
print("distinct genders, raw:           ", raw.select("gender").distinct().count())
print("distinct genders, after trim():  ", raw.select(F.trim("gender")).distinct().count())
print("distinct genders, after regex:   ", raw.select(F.regexp_replace("gender", r"^\s+|\s+$", "")).distinct().count())

Spark's `trim()` removes **spaces but not tabs**. Same idea as Python's `strip()`, different behavior. Never assume two tools agree; check.

The regular expression `^\s+|\s+$` means "any whitespace at the start or the end," and it handles tabs.

The next cell is session 2 in Spark: the whitespace fix, the ID rule (check the text, then cast to BIGINT), the data dictionary, text repair, standardized names, and the coded values. The coded values use **the same crosswalk files** as pandas, joined to the data instead of looked up.

In [ ]:
s2 = raw
for col in raw.columns:
    stripped = F.regexp_replace(F.col(col), r"^\s+|\s+$", "")
    s2 = s2.withColumn(col, F.when(stripped == "", None).otherwise(stripped))

s2 = lspark.ids_to_bigint(s2)                            # every ID checked as 8-digit text, then cast to BIGINT

for col, codes in cleaning.NULL_TOKENS.items():          # the same data dictionary as session 2
    s2 = s2.withColumn(col, F.when(F.upper(col).isin([c.upper() for c in codes]), None).otherwise(F.col(col)))

s2 = lspark.repair_text(s2)                               # the same repair table as session 2

for col in ["last_name", "first_name", "alias"]:          # upper case, accents folded, single spaces
    folded = F.translate(F.upper(col), "ÁÉÍÓÚÑÜ", "AEIOUNU")
    s2 = s2.withColumn(f"{col}_std", F.regexp_replace(folded, r"\s+", " "))

s2 = lspark.apply_domains(s2)                             # the same crosswalk CSV files, joined

lspark.show(s2.filter(F.col("stg_status") != "None")
           .select("tdcj_number", "last_name", "last_name_std", "race", "race_code", "stg_affiliation", "stg_group", "stg_status"))

Spark has no accent-folding function, so `translate` swaps each accented letter this data contains for its plain version. That's a shortcut that only works because we know which letters occur. The reconciliation below is what proves it gives the same answers as pandas.

If you write SQL, Spark SQL does the same work. Here's the gender cleanup as a query:

In [ ]:
raw.createOrReplaceTempView("raw_offenders")
spark.sql(r"""
    SELECT NULLIF(REGEXP_REPLACE(gender, '^\\s+|\\s+$', ''), '') AS gender,
           COUNT(*) AS records
    FROM raw_offenders
    GROUP BY 1
    ORDER BY records DESC
""").show()

🧪 **Your turn #1.** How many records are missing a gender in `s2`? It should match pandas exactly.

In [ ]:
answer_1 = ___      # s2.filter(F.col("gender").isNull()).count()
check.s5_ex1(answer_1)

## 2. Session 3 in Spark: dates
Same shapes as session 3, written in Spark's pattern language (`yyyy` instead of `%Y`). `try_to_timestamp` returns null instead of crashing when a value doesn't fit.

In [ ]:
DATE_PATTERNS = [   # (shape, Spark pattern)
    (r"^\d{2}/\d{2}/\d{4}$", "MM/dd/yyyy"),      (r"^\d{1,2}/\d{1,2}/\d{2}$", "M/d/yy"),
    (r"^\d{2}-\d{2}-\d{4}$", "MM-dd-yyyy"),      (r"^\d{4}-\d{2}-\d{2}$", "yyyy-MM-dd"),
    (r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$", "yyyy-MM-dd HH:mm:ss"),
    (r"^\d{8}$", "yyyyMMdd"),                     (r"^\d{2}-[A-Za-z]{3}-\d{4}$", "dd-MMM-yyyy"),
    (r"^[A-Za-z]+ \d{1,2}, \d{4}$", "MMMM d, yyyy"), (r"^[A-Za-z]{3} \d{1,2} \d{4}$", "MMM d yyyy"),
]
TWO_DIGIT = r"^\d{1,2}/\d{1,2}/\d{2}$"

def parse_date(col):
    value = F.col(col)
    result = None
    for shape, pattern in DATE_PATTERNS:
        parsed = F.to_date(F.try_to_timestamp(value, F.lit(pattern)))
        result = F.when(value.rlike(shape), parsed) if result is None else result.when(value.rlike(shape), parsed)
    return result

spark.createDataFrame([("2/19/64",), ("12/31/99",)], ["text"]).select("text", F.to_date(F.try_to_timestamp("text", F.lit("M/d/yy"))).alias("spark_says")).show()

Spark reads `yy` as **always 2000-2099**, so the Hulk's release lands in 2099 (right, by luck) and Batman is born in 2064 (wrong). Python's default window also put Batman in 2064, but put the Hulk in 1999. Same text, different tools, different defaults, and neither knows what the column means. So we apply **our** rules from session 3, written for Spark:

In [ ]:
def fix_century_past(raw_col, parsed):
    # birth and sentence dates can't be in the future
    return F.when(F.col(raw_col).rlike(TWO_DIGIT) & (parsed > F.current_date()), F.add_months(parsed, -1200)).otherwise(parsed)

def fix_century_after(raw_col, parsed, reference):
    # a release can't come before its sentence; with no sentence date, fall back to Python's 1969-2068 window
    earlier = F.add_months(parsed, -1200)
    with_reference = F.when(earlier >= reference, earlier).otherwise(parsed)
    without_reference = F.when(F.year(parsed) >= 2069, earlier).otherwise(parsed)
    fixed = F.when(reference.isNull(), without_reference).otherwise(with_reference)
    return F.when(F.col(raw_col).rlike(TWO_DIGIT), fixed).otherwise(parsed)

s3 = (s2.withColumn("dob", fix_century_past("date_of_birth", parse_date("date_of_birth")))
        .withColumn("sent", fix_century_past("sentence_date", parse_date("sentence_date"))))
s3 = s3.withColumn("rel", fix_century_after("projected_release_date", parse_date("projected_release_date"), F.col("sent")))
s3 = (s3.drop("date_of_birth", "sentence_date", "projected_release_date")
        .withColumnRenamed("dob", "date_of_birth").withColumnRenamed("sent", "sentence_date")
        .withColumnRenamed("rel", "projected_release_date"))

dob, sent = F.col("date_of_birth"), F.col("sentence_date")
before_birthday = (F.month(sent) < F.month(dob)) | ((F.month(sent) == F.month(dob)) & (F.dayofmonth(sent) < F.dayofmonth(dob)))
s3 = s3.withColumn("age_at_sentencing", F.year(sent) - F.year(dob) - F.when(before_birthday, 1).otherwise(0))
s3 = s3.withColumn("dob_flag",
    F.when(F.year(dob) < 1900, "before 1900").when(dob > F.current_date(), "in the future")
     .when(F.col("age_at_sentencing") < 17, "under 17 at sentencing")
     .when(F.col("age_at_sentencing") > 100, "over 100 at sentencing"))

lspark.show(s3.filter(F.col("dob_flag").isNotNull()).select("alias", "date_of_birth", "age_at_sentencing", "dob_flag"))

Same eight records flagged as in session 3.

## 3. Session 4 in Spark: types

In [ ]:
# Integers: strip "pts", number words -> digits, reject fractions
words = F.create_map([F.lit(x) for pair in conversions.NUMBER_WORDS.items() for x in pair])
points_text = F.trim(F.regexp_replace(F.lower("disciplinary_points"), "pts", ""))
s4 = s3.withColumn("_points_text", F.coalesce(words[points_text], points_text))
s4 = s4.withColumn("_points", F.expr("try_cast(_points_text AS DOUBLE)"))
s4 = s4.withColumn("disciplinary_points", F.when(F.col("_points") == F.floor("_points"), F.col("_points").cast("int")))

# Money: remove $ / USD / commas, (x) means negative, "-" means zero, exact DECIMAL(12,2)
money = F.regexp_replace(F.upper("restitution_owed"), r"USD|\$|,|\s", "")
negative = money.startswith("(") & money.endswith(")")
money = F.regexp_replace(money, r"[()]", "")
s4 = s4.withColumn("_money", F.when(money == "-", F.lit("0")).otherwise(money))
s4 = s4.withColumn("_amount", F.expr("try_cast(_money AS DECIMAL(12,2))"))
s4 = s4.withColumn("restitution_owed", F.when(negative, -F.col("_amount")).otherwise(F.col("_amount")))

# Time of day: same shapes as session 4, stored as HH:mm:ss text (see note below)
TIME_PATTERNS = [(r"^\d{2}:\d{2}$", "HH:mm"), (r"^\d{2}:\d{2}:\d{2}$", "HH:mm:ss"),
                 (r"^\d{1,2}:\d{2} ?[AaPp][Mm]$", "h:mma"), (r"^\d{4}$", "HHmm"), (r"^\d{2}\.\d{2}$", "HH.mm")]
t = F.col("intake_time")
compact = F.upper(F.regexp_replace(t, " ", ""))
intake = None
for shape, pattern in TIME_PATTERNS:
    parsed = F.date_format(F.try_to_timestamp(compact, F.lit(pattern)), "HH:mm:ss")
    intake = F.when(t.rlike(shape), parsed) if intake is None else intake.when(t.rlike(shape), parsed)
s4 = s4.withColumn("intake_time", intake)

# Yes/no, including the single-character Y/N and T/F flags: the same lists as session 4
for col in conversions.FLAG_COLUMNS:
    value = F.upper(col)
    s4 = s4.withColumn(col, F.when(value.isin(list(conversions.TRUE_VALUES)), True)
                             .when(value.isin(list(conversions.FALSE_VALUES)), False))

s4 = s4.drop("_points_text", "_points", "_money", "_amount")
s4.printSchema()

Spark has no time-of-day type on the runtimes you're likely to use, so `intake_time` is stored as validated `HH:mm:ss` text. Spark also has no "category" type; codes stay as text, and the allowed values get enforced with checks instead.

🧪 **Your turn #2.** What's the total restitution owed according to Spark?

In [ ]:
answer_2 = ___      # s4.agg(F.sum("restitution_owed")).first()[0]
check.s5_ex2(answer_2)

## 4. Reconcile: do pandas and Spark agree?
Two pipelines built from the same rules must give the same answers. If they don't, one of them is wrong, and you find out **before** someone builds a report on it.

In [ ]:
import pandas as pd
p4 = pipeline.clean_through(4)

spark_side = s4.agg(
    F.count("*").alias("rows"),
    F.countDistinct("tdcj_number").alias("distinct TDCJ numbers"),
    F.countDistinct("sid_number").alias("distinct SIDs"),
    F.sum("tdcj_number").alias("TDCJ hash total"),
    F.sum(F.col("custody_level").isNull().cast("int")).alias("custody missing"),
    F.sum(F.col("gender").isNull().cast("int")).alias("gender missing"),
    F.sum(F.col("date_of_birth").isNull().cast("int")).alias("birth date missing"),
    F.min("date_of_birth").cast("string").alias("earliest birth date"),
    F.min("projected_release_date").cast("string").alias("earliest release"),
    F.max("projected_release_date").cast("string").alias("latest release"),
    F.sum(F.col("intake_time").isNull().cast("int")).alias("intake time missing"),
    F.sum("disciplinary_points").alias("total points"),
    F.sum("restitution_owed").cast("string").alias("total restitution"),
    F.sum(F.col("escape_risk").cast("int")).alias("escape risk = true"),
    F.sum(F.col("protective_custody").cast("int")).alias("protective custody = true"),
    F.sum(F.col("dampener_required").cast("int")).alias("dampener required = true"),
    F.round(F.avg(F.when(F.col("dob_flag").isNull(), F.col("age_at_sentencing"))), 3).alias("average age"),
    F.sum(F.col("dob_flag").isNotNull().cast("int")).alias("birth dates flagged"),
    F.countDistinct("race_code").alias("distinct race codes"),
    F.sum((F.col("stg_status") == "Confirmed").cast("int")).alias("confirmed STG records"),
    F.sum(lspark.unrecognized_domain_values()).alias("unrecognized coded values"),
    F.sum(lspark.offense_violence_conflict().cast("int")).alias("offense/violence conflicts"),
).first().asDict()

pandas_side = {
    "rows": len(p4), "distinct TDCJ numbers": p4.index.nunique(), "distinct SIDs": p4["sid_number"].nunique(), "TDCJ hash total": p4.index.to_series().sum(),
    "custody missing": p4["custody_level"].isna().sum(), "gender missing": p4["gender"].isna().sum(),
    "birth date missing": p4["date_of_birth"].isna().sum(), "earliest birth date": p4["date_of_birth"].min(),
    "earliest release": p4["projected_release_date"].min(), "latest release": p4["projected_release_date"].max(),
    "intake time missing": p4["intake_time"].isna().sum(), "total points": p4["disciplinary_points"].sum(),
    "total restitution": p4["restitution_owed"].sum(), "escape risk = true": p4["escape_risk"].sum(),
    "protective custody = true": p4["protective_custody"].sum(), "dampener required = true": p4["dampener_required"].sum(),
    "average age": round(float(p4.loc[p4["dob_flag"].isna(), "age_at_sentencing"].mean()), 3),
    "birth dates flagged": p4["dob_flag"].notna().sum(),
    "distinct race codes": p4["race_code"].nunique(), "confirmed STG records": (p4["stg_status"] == "Confirmed").sum(),
    "unrecognized coded values": len(domains.domain_exceptions(p4)),
    "offense/violence conflicts": len(domains.offense_violence_conflicts(p4)),
}
reconcile = pd.DataFrame({"pandas": {k: str(v) for k, v in pandas_side.items()},
                          "Spark": {k: str(v) for k, v in spark_side.items()}})
reconcile["match"] = ["✅" if a == b else "❌" for a, b in zip(reconcile["pandas"], reconcile["Spark"])]
reconcile

Every metric matches, including the restitution total to the cent, the coded values down to the last unrecognized spelling, and the **hash total**, the sum of every TDCJ number. Adding IDs means nothing as a measurement, but as a checksum it proves both sides hold exactly the same records. That's only true because we wrote the century rules ourselves. With each tool's default two-digit-year rule, the birth dates wouldn't agree.

## 5. Save it as a governed table
Spark has no index, so `tdcj_number` stays a column. Before saving, check it's still a real ID: BIGINT, one per record, none missing, all within 8 digits.

In [ ]:
display(lspark.check_ids(s4))      # Spark has no index: prove the ID is still an ID before saving
print(lspark.save_table(s4))

On Databricks (with the catalog set in `lnl/config.py`), this creates a **Delta table in Unity Catalog**, which keeps version history, enforces permissions, and tracks lineage. The table also enforces the IDs: `tdcj_number` is a BIGINT that can't be null, must fit in 8 digits (1 to 99,999,999), and is declared the primary key; `sid_number` must fit in 8 digits when present. That's where cleansed data should live, not in a CSV on someone's desktop. In production, sensitive columns such as race and religion also get restricted, with Unity Catalog column masks or a separate table with tighter permissions. Outside Databricks it writes Parquet files instead.

## Wrap-up
- The same rules port from pandas to Spark, but **functions with the same name can behave differently** (`trim`, two-digit years).
- IDs follow one rule in both tools: check the text, then convert to BIGINT on purpose.
- Reference data is shared: both engines read the same crosswalk files, so one fix to a crosswalk fixes both.
- Spark SQL is a full alternative if you think in SQL.
- **Reconcile** any two implementations before trusting either one.

**Next session:** text. Finding "good conduct" in 10,000 free-text officer notes.

---
## Answer key
```python
answer_1 = s2.filter(F.col("gender").isNull()).count()
answer_2 = s4.agg(F.sum("restitution_owed")).first()[0]
```